## NAME: Souri Rishik Volety
## Reg No: 230968004

### Recursive Best-First Search (RBFS) for Maze Navigation
Problem Statement:
Consider a maze represented as a graph, where each node corresponds to a location in the maze
and edges represent valid paths between locations.
An agent is placed at a start node and must reach a goal node by traversing the maze.
Traditional A* requires storing all explored nodes in memory, which may become infeasible for large
mazes.
Recursive Best-First Search (RBFS) is a memory-bounded heuristic search that uses recursion and
limited memory while still aiming for optimal solutions.
Task:
1. Formulate the maze navigation problem as a search problem by clearly defining: State space,
Initial state, Actions, Transition model, Goal test, Path cost, Heuristic function ℎ(𝑛)
2. Implement the Recursive Best-First Search (RBFS) algorithm to find an optimal path from
the start node to the goal node.
3. Use the evaluation function:𝑓(𝑛)=𝑔(𝑛)+ℎ(𝑛)
where:
𝑔(𝑛) = path cost from start to node 𝑛
ℎ(𝑛) = Manhattan distance from node 𝑛 to the goal
1. Print the path returned by RBFS.
2. Count and display the number of nodes expanded during the search.

Constraints:
• The maze is unweighted (each step cost = 1).
• The environment is fully observable, deterministic, static, discrete, and single-agent.
• RBFS must be implemented without using built-in graph search libraries.
• The algorithm should use only linear memory through recursion.

In [1]:
import math

class MazeRBFS:
    def __init__(self, maze, start, goal):
        self.maze = maze
        self.start = start
        self.goal = goal
        self.nodes_expanded = 0
        self.rows = len(maze)
        self.cols = len(maze[0])

    def get_neighbors(self, node):
        x, y = node
        neighbors = []
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.rows and 0 <= ny < self.cols and self.maze[nx][ny] == 0:
                neighbors.append((nx, ny))
        return neighbors

    def manhattan_distance(self, node):
        return abs(node[0] - self.goal[0]) + abs(node[1] - self.goal[1])

    def search(self):
        result, f_final, path = self.rbfs(self.start, g=0, f_limit=math.inf, path=[self.start])
        return path, self.nodes_expanded

    def rbfs(self, node, g, f_limit, path):
        if node == self.goal:
            return node, f_limit, path

        neighbors = self.get_neighbors(node)
        if not neighbors:
            return None, math.inf, None

        self.nodes_expanded += 1

        successors = []
        for snode in neighbors:
            if snode in path: continue
            new_g = g + 1
            f_val = max(new_g + self.manhattan_distance(snode), new_g + self.manhattan_distance(node)) 
            successors.append({'node': snode, 'f': f_val, 'g': new_g})

        if not successors:
            return None, math.inf, None

        while True:
            successors.sort(key=lambda x: x['f'])
            best = successors[0]
            
            if best['f'] > f_limit:
                return None, best['f'], None
            
            alternative = successors[1]['f'] if len(successors) > 1 else f_limit
            
            res_node, res_f, res_path = self.rbfs(best['node'], best['g'], min(f_limit, alternative), path + [best['node']])
            
            best['f'] = res_f
            
            if res_node is not None:
                return res_node, res_f, res_path

In [2]:
maze_grid = [
    [0, 0, 0, 0],
    [1, 1, 0, 1],
    [0, 0, 0, 0],
    [0, 1, 1, 0]
]
start_pos = (0, 0)
goal_pos = (3, 3)

solver = MazeRBFS(maze_grid, start_pos, goal_pos)
path, expanded = solver.search()

print(f"Path Found: {path}")
print(f"Nodes Expanded: {expanded}")

Path Found: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (3, 3)]
Nodes Expanded: 6


### Iterative Deepening A* (IDA*) for Maze Navigation
Problem Statement:
Consider a maze represented as a graph, where each node corresponds to a location in the maze
and edges represent valid paths between locations.
An agent must reach the goal node efficiently.
While A* guarantees optimality, it requires large memory.
Iterative Deepening A* (IDA*) overcomes this by combining Depth-first traversal, A* heuristic cost
function, Iterative deepening on cost limits, IDA* is especially useful when memory is limited.
Task:
1. Formulate the maze navigation problem as a search problem by clearly defining: State space,
Initial state, Actions, Transition model, Goal test, Path cost, Heuristic function h(n)
2. Implement the Iterative Deepening A* (IDA*) algorithm to find the shortest path from the
start node to the goal node.
3. Use the cost function: f(n)=g(n)+h(n)
4. Perform iterative deepening by increasing the threshold value until a solution is found.
5. Print the path returned by IDA*.
6. Count and display the number of nodes expanded during the search.

Constraints:
• The maze is unweighted (uniform cost = 1).
• The heuristic must be admissible.
• The environment is fully observable, deterministic, static, discrete, and single-agent.
• IDA* must be implemented without built-in search libraries.
• The algorithm must use depth-first search with cost-bounded iterations.

In [3]:
class MazeIDAStar:
    def __init__(self, maze, start, goal):
        self.maze = maze
        self.start = start
        self.goal = goal
        self.nodes_expanded = 0
        self.rows = len(maze)
        self.cols = len(maze[0])

    def h(self, node):
        return abs(node[0] - self.goal[0]) + abs(node[1] - self.goal[1])

    def get_neighbors(self, node):
        x, y = node
        neighbors = []
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < self.rows and 0 <= ny < self.cols and self.maze[nx][ny] == 0:
                neighbors.append((nx, ny))
        return neighbors

    def search(self):
        threshold = self.h(self.start)
        path = [self.start]
        
        while True:
            result, t = self.dfs_recursive(path, 0, threshold)
            
            if result == "FOUND":
                return path, self.nodes_expanded
            if t == float('inf'):
                return None, self.nodes_expanded
            
            threshold = t  

    def dfs_recursive(self, path, g, threshold):
        node = path[-1]
        f = g + self.h(node)
        
        if f > threshold:
            return "NOT_FOUND", f
        
        if node == self.goal:
            return "FOUND", f
        
        self.nodes_expanded += 1
        min_threshold = float('inf')
        
        for neighbor in self.get_neighbors(node):
            if neighbor not in path:
                path.append(neighbor)
                res, t = self.dfs_recursive(path, g + 1, threshold)
                
                if res == "FOUND":
                    return "FOUND", t
                
                if t < min_threshold:
                    min_threshold = t
                
                path.pop()
                
        return "NOT_FOUND", min_threshold

In [5]:
maze_grid = [
    [0, 0, 0, 0],
    [1, 1, 0, 1],
    [0, 0, 0, 0],
    [0, 1, 1, 0]
]

start_node = (0, 0)
goal_node = (3, 3)

solver = MazeIDAStar(maze_grid, start_node, goal_node)
final_path, total_expanded = solver.search()

print(f"IDA* SEARCH RESULTS")
print(f"Final Path: {final_path}")
print(f"Nodes Expanded: {total_expanded}")

IDA* SEARCH RESULTS
Final Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (3, 3)]
Nodes Expanded: 6


## Simplified Memory-Bounded A* (SMA*) for Maze Navigation

Problem Statement:
Consider a maze represented as a graph, where each node corresponds to a location in the maze
and edges represent valid paths between locations.
An agent must reach the goal node using an optimal strategy.
A* search may run out of memory when the search space becomes large.
Simplified Memory-Bounded A* (SMA*) is a variant of A* that uses a fixed memory limit by
Expanding best nodes first, Dropping worst leaf nodes when memory is full, Retaining optimality
when enough memory is available
Task:
1. Formulate the maze navigation problem as a search problem by clearly defining: State
space, Initial state, Actions, Transition model, Goal test, Path cost, Heuristic function h(n)
2. Implement the Simplified Memory-Bounded A* (SMA*) algorithm to find the shortest path
from the start node to the goal node.
3. Use the evaluation function: f(n)=g(n)+h(n)
4. Introduce a fixed memory limit (maximum number of nodes stored).
5. When memory is full:
• Remove the node with the highest f(n)f(n)f(n) value
• Backup its cost to its parent
6. Print the final path returned by SMA*.
7. Count and display the number of nodes expanded during the search.

Constraints:
• Uniform step cost = 1.
• The heuristic must be admissible.
• The environment is fully observable, deterministic, static, discrete, and single-agent.
• SMA* must be implemented without built-in graph search libraries.
• The algorithm must respect a predefined memory limit.

In [6]:
import math

class Node:
    def __init__(self, state, parent=None, g=0, h=0):
        self.state = state
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g + h
        self.children = {}
        self.forgotten = math.inf

class SMAStar:
    def __init__(self, maze, start, goal, max_memory):
        self.maze = maze
        self.start = start
        self.goal = goal
        self.max_memory = max_memory
        self.nodes_expanded = 0
        self.memory = {}

    def get_neighbors(self, state):
        x, y = state
        results = []
        for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nx, ny = x + dx, y + dy
            if 0 <= nx < len(self.maze) and 0 <= ny < len(self.maze[0]) and self.maze[nx][ny] == 0:
                results.append((nx, ny))
        return results

    def manhattan(self, state):
        return abs(state[0] - self.goal[0]) + abs(state[1] - self.goal[1])

    def solve(self):
        root = Node(self.start, h=self.manhattan(self.start))
        self.memory[self.start] = root
        
        while True:
            if not self.memory: return None, self.nodes_expanded
            
            best_node = self.find_best_leaf(root)
            
            if best_node.state == self.goal:
                return self.reconstruct_path(best_node), self.nodes_expanded

            successors = self.get_neighbors(best_node.state)
            self.nodes_expanded += 1
            
            for s_state in successors:
                if self.is_in_path(best_node, s_state): continue
                
                g = best_node.g + 1
                h = self.manhattan(s_state)
                f = max(best_node.f, g + h)
                
                if len(self.memory) >= self.max_memory:
                    if not self.evict_worst_node(root): break
                
                successor_node = Node(s_state, best_node, g, h)
                successor_node.f = f
                best_node.children[s_state] = successor_node
                self.memory[s_state] = successor_node

            if not best_node.children:
                best_node.f = math.inf
            else:
                best_node.f = min(min(c.f for c in best_node.children.values()), best_node.forgotten)
            
            self.update_parents(best_node)

    def find_best_leaf(self, node):
        if not node.children: return node
        best_child = min(node.children.values(), key=lambda n: (n.f, -n.g))
        return self.find_best_leaf(best_child)

    def evict_worst_node(self, root):
        worst_node = self.find_worst_leaf(root)
        if worst_node == root: return False
        parent = worst_node.parent
        if parent:
            parent.forgotten = min(parent.forgotten, worst_node.f)
            del parent.children[worst_node.state]
        if worst_node.state in self.memory:
            del self.memory[worst_node.state]
        return True

    def find_worst_leaf(self, node):
        if not node.children: return node
        worst_child = max(node.children.values(), key=lambda n: (n.f, n.g))
        return self.find_worst_leaf(worst_child)

    def is_in_path(self, node, state):
        curr = node
        while curr:
            if curr.state == state: return True
            curr = curr.parent
        return False

    def update_parents(self, node):
        curr = node.parent
        while curr:
            old_f = curr.f
            new_f = min(min(c.f for c in curr.children.values()), curr.forgotten) if curr.children else curr.forgotten
            if old_f == new_f: break
            curr.f = new_f
            curr = curr.parent

    def reconstruct_path(self, node):
        path = []
        while node:
            path.append(node.state)
            node = node.parent
        return path[::-1]

In [7]:
maze = [
    [0, 0, 0, 0],
    [1, 1, 0, 1],
    [0, 0, 0, 0],
    [0, 1, 1, 0]
]

solver = SMAStar(maze, (0, 0), (3, 3), max_memory=10)
path, expanded = solver.solve()
print(f"Path: {path}")
print(f"Expanded: {expanded}")

Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (3, 3)]
Expanded: 6
